# Day 16 — Practice Session · **SOLUTIONS**
### Filtering, Selection & Indexing · Python for Data Science

**Prepared by Srinivasa Sai Chava**  ·  Boston University

---

> **Instructor copy.** Every question is followed by the answer and the reasoning.
> The student copy (`Day16_Practice_Questions.ipynb`) is identical minus the answer blocks.

| Part | Focus | Questions |
|---|---|---|
| A | Predict the output | 8 |
| B | Spot & fix the bug | 4 |
| C | Write the code | 5 |
| D | Challenge | 2 |

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore")

df = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran", "Priya"],
    "city":   ["Pune", "Mumbai", "Delhi", "Pune", "Delhi", "Mumbai"],
    "python": [88, 91, 45, 67, 72, 58],
    "stats":  [71, 84, 38, 73, 69, 62],
})
print("pandas", pd.__version__)
df

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore")

df = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran", "Priya"],
    "city":   ["Pune", "Mumbai", "Delhi", "Pune", "Delhi", "Mumbai"],
    "python": [88, 91, 45, 67, 72, 58],
    "stats":  [71, 84, 38, 73, 69, 62],
})
print("pandas", pd.__version__)
df

---
# Part A — Predict the Output  *(8 min)*

---
# Part A — Predict the Output

**Teaching note:** A3 (the slicing difference) and A6 (the filtered index) are the two that
matter most. A6 is the single most valuable question in the notebook.

### A1. Do these agree?

In [ ]:
print(df.loc[1, "name"])
print(df.iloc[1, 0])

*Your prediction:*  

> ### ✅ Answer A1
> ```
> Sara
> Sara
> ```
> **They agree — and that is exactly the trap.** On a fresh DataFrame the labels *are*
> 0, 1, 2, … so `loc[1]` (the row *labelled* 1) and `iloc[1]` (the *second* row) point at the
> same thing.
>
> Because of this you can write either one for weeks without learning the difference. Then
> you filter, and half your code quietly breaks. A6 shows exactly that.

### A2. What happens with -1?

In [ ]:
print(df.iloc[-1]["name"])
print(df.loc[-1])

*Your prediction:*  

> ### ✅ Answer A2
> ```
> Priya
> KeyError: -1
> ```
> **Why:** `iloc[-1]` counts backwards from the end, exactly like a Python list — Day 4.
>
> `loc[-1]` asks for the row *labelled* −1, and there is no such label. Negative indexing is
> a **position** idea; it has no meaning in a world of labels.

### A3. How many rows does each give?

In [ ]:
print(len(df.loc[1:3]))
print(len(df.iloc[1:3]))

*Your prediction:*  

> ### ✅ Answer A3
> ```
> 3
> 2
> ```
> **`loc` slicing INCLUDES the stop; `iloc` excludes it.** `loc[1:3]` gives labels 1, 2 and 3.
> `iloc[1:3]` gives positions 1 and 2.
>
> **Why this is not really an inconsistency:** `iloc` counts, so it follows Python's rule
> from Day 4 — stop excluded, like `range()`. `loc` names a first and last row, and with
> names there is no "next one" to stop before. Asking for *"Ravi to Neha"* obviously means
> Neha included.

### A4. What is selected?

In [ ]:
print(df[(df["python"] > 70) & (df["stats"] > 70)]["name"].tolist())
print(df[~(df["city"] == "Pune")]["name"].tolist())

*Your prediction:*  

> ### ✅ Answer A4
> ```
> ['Ravi', 'Sara']
> ['Sara', 'Amit', 'Kiran', 'Priya']
> ```
> **Why:** the first needs **both** conditions true. Ravi (88, 71) and Sara (91, 84) qualify;
> Neha has 67 in python and Kiran has 69 in stats, so both fail.
>
> `~` inverts the mask, keeping everyone not from Pune. Same `&` `|` `~` rules as Day 12 and
> Day 13 — this is their third appearance.

### A5. Which names come back?

In [ ]:
print(df[df["python"].between(58, 88)]["name"].tolist())
print(df[df["city"].isin(["Delhi", "Mumbai"])]["name"].tolist())

*Your prediction:*  

> ### ✅ Answer A5
> ```
> ['Ravi', 'Neha', 'Kiran', 'Priya']
> ['Sara', 'Amit', 'Kiran', 'Priya']
> ```
> **Why:** `between` is **inclusive at both ends**, so Priya (58) and Ravi (88) — the exact
> boundary values — are both kept. That is unusual: `range()`, slicing and `np.arange` all
> exclude the stop.
>
> If a boundary matters, pass `inclusive="neither"`, `"left"` or `"right"`.
>
> `isin` is a cleaner way to write a chain of `==` comparisons joined by `|`.

### A6. What does each line do?

In [ ]:
top = df[df["python"] > 70]
print(top.index.tolist())
print(top.iloc[2]["name"])
print(top.loc[2])

*Your prediction:*  

> ### ✅ Answer A6
> ```
> [0, 1, 4]
> Kiran
> KeyError: 2
> ```
> **This is the most important question in the notebook.**
>
> Filtering **keeps the original labels**. The three surviving rows are Ravi, Sara and Kiran,
> which were labels 0, 1 and 4 — so the labels now have a **gap**, while the positions are
> 0, 1, 2.
>
> - `top.iloc[2]` → the *third row* → Kiran ✓
> - `top.loc[2]` → the row *labelled 2* → that was Amit, who was filtered out → `KeyError`
>
> **A `KeyError` on a row you can see on screen** is the symptom. The fix is
> `reset_index(drop=True)` whenever the old numbering has stopped meaning anything.

### A7. Did the value change?

In [ ]:
d = df.copy()
d[d["python"] < 50]["python"] = 50
print(d.loc[2, "python"])

*Your prediction:*  

> ### ✅ Answer A7
> ```
> 45
> ```
> **Nothing changed.** Two sets of brackets in a row means pandas built a temporary **copy**
> of the filtered rows, wrote 50 into the copy, and threw the copy away.
>
> Modern pandas warns about this. Older versions sometimes worked and sometimes did not,
> which was worse — the same code could behave differently on two machines.
>
> **The fix is one `.loc`:** `d.loc[d["python"] < 50, "python"] = 50`
>
> This is Day 12's copy-versus-view question in a new costume. There, a NumPy slice was a
> *view* and surprised you by changing the original. Here a filtered DataFrame is a *copy*
> and surprises you by **not** changing it.

### A8. What does the index become?

In [ ]:
named = df.set_index("name")
print(named.loc["Sara", "python"])
print(named.iloc[0].name)
print(named.loc["Ravi":"Amit"].index.tolist())

*Your prediction:*  

> ### ✅ Answer A8
> ```
> 91
> Ravi
> ['Ravi', 'Sara', 'Amit']
> ```
> **Why:** after `set_index("name")`, the names are the labels, so `loc` reads beautifully —
> `named.loc["Sara", "python"]`.
>
> **`iloc` is unaffected.** It always means positions, whatever the index holds, so
> `iloc[0]` is still the first row.
>
> The label slice returns three rows because `loc` includes the stop — and here that feels
> obviously right: *Ravi to Amit* means Amit included.

---
# Part B — Spot & Fix the Bug  *(7 min)*

---
# Part B — Spot & Fix the Bug

**Teaching note:** B1 and B2 are the two that cost real debugging time. Neither raises a
useful error.

### B1. This should cap every mark above 90 at 90.

In [ ]:
d = df.copy()
d[d["python"] > 90]["python"] = 90
print(d[["name", "python"]])

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B1
> **Symptom:** Sara still has 91. The code ran, printed no error, and did nothing.
> **Cause:** chained assignment — `d[mask]["python"]` builds a copy, and the 90 is written
> into the copy.

In [ ]:
d = df.copy()
print("before:", d.loc[1, "python"])

d.loc[d["python"] > 90, "python"] = 90        # ONE .loc: rows AND column

print("after :", d.loc[1, "python"])
print(d[["name", "python"]])

# THE RULE: to change data, use one  .loc[rows, column]
# Never two square brackets in a row.

### B2. This should print the third student in the filtered table.

In [ ]:
passed = df[df["python"] >= 58]
print(passed[["name", "python"]])
print(passed.loc[2])

*Error:* 

*Your fix:*

> ### ✅ Answer B2
> **Error:** `KeyError: 2`
> **Cause:** Amit (label 2) scored 45 and was filtered out. The filtered table keeps the
> original labels — `[0, 1, 3, 4, 5]` — so label 2 no longer exists, even though there is
> visibly a third row on screen.

In [ ]:
passed = df[df["python"] >= 58]
print("labels   :", passed.index.tolist())      # [0, 1, 3, 4, 5]
print("positions: 0, 1, 2, 3, 4")
print()

# Option 1 - ask by POSITION
print("iloc[2]:", passed.iloc[2]["name"])       # Neha

# Option 2 - renumber, then labels and positions agree again
passed = passed.reset_index(drop=True)
print("after reset, loc[2]:", passed.loc[2, "name"])

# Printing .index after any filter is a habit worth building.

### B3. This should select students scoring 60-80.

In [ ]:
print(df[df["python"] > 60 and df["python"] < 80]["name"].tolist())

*Error:* 

*Your fix:*

> ### ✅ Answer B3
> **Error:** `ValueError: The truth value of a Series is ambiguous`
> **Cause:** `and` needs a single `True`/`False` on each side, but each condition is a Series
> of six booleans.

In [ ]:
print(df[(df["python"] > 60) & (df["python"] < 80)]["name"].tolist())
# ['Neha', 'Kiran']

# Or more readably for a range:
print(df[df["python"].between(61, 79)]["name"].tolist())

# And the brackets are NOT optional - & binds tighter than >, so without them
# Python tries to evaluate  60 & df["python"]  first:
try:
    df[df["python"] > 60 & df["python"] < 80]
except Exception as e:
    print(f"\nwithout brackets -> {type(e).__name__}")

### B4. This should give the last two rows.

In [ ]:
print(df.loc[-2:])

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B4
> **Symptom:** you get **all six rows**, not the last two — and no error whatsoever.
> **Cause:** `loc` slices by label. The index is sorted, so pandas treats the label −2 as
> sitting *before* label 0 and slices from the very beginning to the end.
>
> **This is worse than an error.** You asked for two rows, received the entire table, and
> nothing complained. Any count or average computed from it would be silently wrong.

In [ ]:
print("loc[-2:]  gave", len(df.loc[-2:]), "rows:", df.loc[-2:]["name"].tolist())
print("iloc[-2:] gave", len(df.iloc[-2:]), "rows:", df.iloc[-2:]["name"].tolist())
print()
print(df.iloc[-2:][["name", "python"]])        # the correct answer

# WHY loc behaved that way: with a sorted numeric index, a label slice does
# not require the endpoints to exist - pandas finds where they WOULD sit.
# -2 sits before 0, so the slice starts at the first row.
#
# Note it depends on the index. With names as the index, the same slice
# raises instead:
named = df.set_index("name")
try:
    named.loc[-2:]
except TypeError as e:
    print("
on a string index, loc[-2:] ->", type(e).__name__)

# Counting from the end? That is ALWAYS iloc.

---
# Part C — Write the Code  *(12 min)*

---
# Part C — Write the Code

**Teaching note:** C4 is the one to demonstrate — assignment through `.loc` is the pattern
they will use constantly and get wrong most often.

### C1. Basic selection
Print:
1. the row at position 3, using `iloc`
2. Neha's python mark, by setting the name as the index
3. the first three rows, names and python only

In [ ]:
# your code here

In [ ]:
# 1. by POSITION
print(df.iloc[3])
print()

# 2. by LABEL, after making the names the index
named = df.set_index("name")
print("Neha's python:", named.loc["Neha", "python"])      # 67
print()

# 3. first three rows, two columns
print(df.iloc[:3][["name", "python"]])
# or equivalently, mixing label and position:
print(df.loc[:2, ["name", "python"]])      # note: loc INCLUDES label 2

### C2. Filter with one condition
Print the names of everyone who scored **60 or more** in stats, and how many that is.

In [ ]:
# your code here

In [ ]:
good = df[df["stats"] >= 60]

print(good["name"].tolist())
print("count:", len(good))

# ['Ravi', 'Sara', 'Neha', 'Kiran', 'Priya']
# count: 5
#
# len(filtered) is the quickest way to count matches. So is
# (df["stats"] >= 60).sum(), since True counts as 1 - Day 1.
print("also:", (df["stats"] >= 60).sum())

### C3. Combine conditions
Print the names of students who are **from Mumbai or Delhi** *and* scored **above 55 in
python**.

In [ ]:
# your code here

In [ ]:
sel = df[df["city"].isin(["Mumbai", "Delhi"]) & (df["python"] > 55)]
print(sel[["name", "city", "python"]])

#     name    city  python
# 1   Sara  Mumbai      91
# 4  Kiran   Delhi      72
# 5  Priya  Mumbai      58
#
# Amit is from Delhi but scored 45, so he is excluded.
#
# The same thing with query(), where 'and' IS allowed:
print()
print(df.query("city in ['Mumbai', 'Delhi'] and python > 55")["name"].tolist())

### C4. Change values safely
Every student below 50 in python must be given a resit mark of exactly 50. Do it **without**
chained assignment, and prove the change took effect.

In [ ]:
d = df.copy()

# your code here

In [ ]:
d = df.copy()

print("before:", d[["name", "python"]].to_dict("records")[2])

d.loc[d["python"] < 50, "python"] = 50        # ONE .loc: rows AND column

print("after :", d[["name", "python"]].to_dict("records")[2])
print()
print("anyone still below 50?", (d["python"] < 50).any())

# before: {'name': 'Amit', 'python': 45}
# after : {'name': 'Amit', 'python': 50}
# anyone still below 50? False
#
# Checking with .any() afterwards is a cheap way to prove an assignment
# actually happened - which chained assignment would not have done.

### C5. Sort and re-index
Produce a table of the top four python scorers, sorted highest first, with the index
renumbered 0–3.

In [ ]:
# your code here

In [ ]:
top4 = (df.sort_values("python", ascending=False)
          .head(4)
          .reset_index(drop=True))

print(top4[["name", "python"]])

#     name  python
# 0   Sara      91
# 1   Ravi      88
# 2  Kiran      72
# 3   Neha      67
#
# nlargest does the sort-and-take in one call:
print()
print(df.nlargest(4, "python").reset_index(drop=True)[["name", "python"]])
#
# Without reset_index the labels would be 1, 0, 4, 3 - the ORIGINAL row
# numbers in their new order, which is confusing in a report.

---
# Part D — Challenge  *(3 min, or take home)*

---
# Part D — Challenge

**Teaching note:** D2 is the one that proves whether A6 landed. If they can explain why the
two answers differ, the session worked.

### D1. A shortlist report
Produce a report of students who scored **above the class average** in python, showing name,
city, python mark, and how far above average they were. Sort by that gap, biggest first, and
renumber the index.

In [ ]:
# your code here

In [ ]:
avg = df["python"].mean()
print(f"class average: {avg:.2f}")

above = df[df["python"] > avg].copy()          # .copy() - we are about to add a column
above["above_by"] = (above["python"] - avg).round(2)

report = (above.sort_values("above_by", ascending=False)
               .reset_index(drop=True)
               [["name", "city", "python", "above_by"]])
print()
print(report)

# class average: 70.17
#
#     name    city  python  above_by
# 0   Sara  Mumbai      91     20.83
# 1   Ravi    Pune      88     17.83
# 2  Kiran   Delhi      72      1.83
#
# TWO details worth naming:
#   .copy() after filtering - without it, adding a column to a filtered
#     view triggers a warning and may not behave as you expect.
#   reset_index(drop=True) - so the report reads 0, 1, 2 rather than
#     carrying the original row numbers 1, 0, 4.

### D2. Prove the trap
Filter the table to students scoring above 60 in python. Then, **using the same number 3**,
get two different students out of the result — one with `loc` and one with `iloc`. Explain
why they differ.

In [ ]:
# your code here

In [ ]:
good = df[df["python"] > 60]
print(good[["name", "python"]])
print()
print("labels   :", good.index.tolist())
print("positions: 0, 1, 2, 3")
print()

print("good.loc[3] ->", good.loc[3, "name"])     # the row LABELLED 3
print("good.iloc[3] ->", good.iloc[3]["name"])   # the FOURTH row

# labels   : [0, 1, 3, 4]
# positions: 0, 1, 2, 3
#
# good.loc[3]  -> Neha    (label 3 survived the filter)
# good.iloc[3] -> Kiran   (the 4th row in the result)
#
# WHY: Amit (label 2) scored 45 and was removed, so from that point on the
# labels run ahead of the positions. The same number 3 means two different
# rows depending on which language you ask in.
#
# On the ORIGINAL df they would agree - df.loc[3] and df.iloc[3] are both
# Neha. Filtering is what breaks the match.
print()
print("on the original df:", df.loc[3, "name"], "and", df.iloc[3]["name"])

# reset_index makes them agree again:
tidy = good.reset_index(drop=True)
print("after reset:", tidy.loc[3, "name"], "and", tidy.iloc[3]["name"])

---
## Done? Self-check

- [ ] I can say instantly whether a task needs `loc` or `iloc`
- [ ] I know `loc` slicing includes the stop and `iloc` does not
- [ ] I know why `df.loc[-1]` fails
- [ ] I know why `df[mask]["col"] = v` does nothing
- [ ] I know why `loc[2]` can fail on a row I can see
- [ ] I printed `.index` after filtering at least once
- [ ] I know `between` includes both endpoints

### Homework
1. Filter a dataset, then show `loc` and `iloc` disagreeing on the same number.
2. Write five questions about a CSV and answer each with one filter.
3. Set a meaningful index on a dataset and look up three rows by name.

### Next class — Topic 1.16: GroupBy & aggregation
Split the table into groups, compute a summary for each, and put the answers back together.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*

---
## Wrap-up — running the last 5 minutes

Three cold-call questions:

1. *"`df.loc[1:3]` — how many rows?"* → three. `loc` includes the stop.
2. *"Why does `df[mask]["col"] = 5` do nothing?"* → it writes to a copy.
3. *"After filtering, why can `loc[2]` fail?"* → label 2 was removed; positions renumbered.

**Common misconceptions to watch for today**

| Misconception | Correction |
|---|---|
| "`loc` and `iloc` are interchangeable" | They agree only while labels equal positions |
| "Slicing always excludes the stop" | `loc` **includes** it; `iloc` excludes it |
| "`loc[-1]` gives the last row" | `KeyError`. Negative indexing is positional |
| "`loc[-2:]` gives the last two" | On a numeric index it returns **everything**, silently |
| "Filtering renumbers the rows" | It keeps the **original** labels, gaps and all |
| "`df[mask]["col"] = v` works" | It writes to a copy and is discarded |
| "`between(60, 90)` means 61–89" | Both endpoints are **included** |
| "`reset_index()` is enough" | Without `drop=True` the old labels become a column |

**A6 and D2 are the same lesson twice**, deliberately. If a student can explain D2 in their
own words, they have understood the day. If they cannot, have them print `.index` after
every filter for the next week — the habit teaches the concept faster than any explanation.

**On the chained assignment**, the honest framing is that it is unreliable rather than
merely wrong: older pandas sometimes worked, which is how the myth survives that it is fine.
Recent versions make it fail consistently, which is an improvement.

**Homework given:** demonstrate `loc`/`iloc` disagreeing; five questions answered by five
filters; set and use a meaningful index.

**Next session:** Topic 1.16 — GroupBy. It replaces the loop-over-cities pattern from Day
15's practice with a single line, and it is where pandas starts to feel powerful rather than
fussy.